In [6]:
%%writefile HPC_1A.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <omp.h>

using namespace std;

const int MAX = 100;

vector<int> graph[MAX];
bool visited[MAX];

// Parallel BFS for Graph
void parallelBFS(int start) {

    queue<int> q;

    visited[start] = true;
    q.push(start);

    while (!q.empty()) {

        int size = q.size();

        #pragma omp parallel for
        for (int i = 0; i < size; i++) {

            int curr;

            // Critical section for queue access
            #pragma omp critical
            {
                curr = q.front();
                q.pop();

                cout << curr << " ";
            }

            // Traverse adjacent vertices
            for (int j = 0; j < graph[curr].size(); j++) {

                int neighbor = graph[curr][j];

                if (!visited[neighbor]) {

                    #pragma omp critical
                    {
                        if (!visited[neighbor]) {
                            visited[neighbor] = true;
                            q.push(neighbor);
                        }
                    }
                }
            }
        }
    }
}

int main() {

    int vertices, edges;

    cout << "Enter number of vertices: ";
    cin >> vertices;

    cout << "Enter number of edges: ";
    cin >> edges;

    cout << "Enter edges (u v):\n";

    for (int i = 0; i < edges; i++) {

        int u, v;
        cin >> u >> v;

        graph[u].push_back(v);
        graph[v].push_back(u); // Undirected graph
    }

    int start;

    cout << "Enter starting vertex: ";
    cin >> start;

    cout << "\nParallel BFS Traversal: ";

    parallelBFS(start);

    return 0;
}

Overwriting HPC_1A.cpp


In [11]:
!g++ HPC_1A.cpp -o HPC_1A

In [12]:
!./HPC_1A


Enter number of vertices: 5
Enter number of edges: 4
Enter edges (u v):
0 1
0 2
1 3
1 4
Enter starting vertex: 0

Parallel BFS Traversal: 0 1 2 3 4 

In [ ]:
5

## Detailed Viva Notes: HPC_1A (Parallel BFS using OpenMP)

This notebook creates and runs a C++ program for **parallel Breadth-First Search (BFS)** on an undirected graph.

### Program 1: Write C++ source (`HPC_1A.cpp`)

- Key components:
  - Graph stored as adjacency list `vector<int> graph[MAX]`
  - `visited[]` tracks discovered vertices
  - `parallelBFS(start)` processes one BFS level at a time
- Parallelization:
  - `#pragma omp parallel for` over current queue size
  - `#pragma omp critical` protects queue pop/push and print
- Why critical needed: queue is shared; unsynchronized access causes race conditions.

**Example input**

- Vertices: `5`
- Edges: `4`
- Edge list: `(0 1), (0 2), (1 3), (1 4)`
- Start vertex: `0`

**Expected output behavior**

- Displays BFS traversal from start node, such as:
  - `Parallel BFS Traversal: 0 1 2 3 4`
- Order of nodes in same level may vary slightly due to parallel scheduling.

### Program 2: Compile C++ program

- Command compiles source into executable.
- Expected output: no errors if compiler and OpenMP support are available.

### Program 3: Run executable

- Program asks user for graph input interactively.
- Expected output:
  - Prompts for vertices/edges/start
  - Final BFS traversal line

### Program 4: Extra standalone code cell

- Contains only `5` (not connected to executable input stream automatically).
- Viva note: this cell alone does not feed input to `./HPC_1A` unless notebook input redirection is used.

---

## Viva Quick Answers

1. BFS vs DFS? BFS explores level-by-level using queue.
2. Why queue critical section? Shared mutable data among threads.
3. Why traversal order may differ? Parallel thread scheduling is non-deterministic.
4. Is result still valid BFS? Yes, level-wise structure is preserved though sibling order can vary.
